# 03 — Training-only exploration and model-specification assessment

This notebook reproduces every statistical calculation used in the stage report. It immediately isolates the 40,000 training records and never summarizes, plots, or models the 10,000 test outcomes.

## Questions answered

1. What do the training distributions and tails look like?
2. How are fare and the proposed predictors associated?
3. Is multicollinearity severe?
4. What does a diagnostic OLS fit reveal?
5. Do linearity, constant variance, normality, and influence assumptions look credible?
6. What specification should be frozen before test evaluation?

In [1]:
from pathlib import Path
import hashlib

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from scipy import stats

from src.taxi_lts.explore_training import (
    PREDICTORS, TARGET, coefficient_table, correlation_tables,
    descriptive_statistics, diagnostic_tables, fit_diagnostic_ols,
    load_training_data, vif_table,
)

PROJECT_ROOT = Path.cwd()
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 40)

## 1. Lock the analytical scope to training data

`load_training_data` filters to `split == "train"` immediately and checks that exactly 40,000 unique source IDs remain. We retain raw units for interpretation; scaling will be learned from training data only during model fitting.

In [2]:
input_path = PROJECT_ROOT / "data/processed/taxi_modeling_50000.csv"
training = load_training_data(input_path)

assert len(training) == 40_000
assert training["split"].eq("train").all()
assert training["source_row_id"].is_unique

fingerprint = hashlib.sha256(training["source_row_id"].to_numpy(dtype="int64").tobytes()).hexdigest()
print("Training observations:", f"{len(training):,}")
print("Training source-ID fingerprint:", fingerprint)
print("Predictors:", PREDICTORS)

Training observations: 40,000
Training source-ID fingerprint: 199b0d9697584054d47e44638848bc40fb1b2467dbf07902991466df097eb384
Predictors: ['trip_distance', 'duration_min', 'passenger_count', 'pickup_hour_sin', 'pickup_hour_cos', 'is_weekend']


## 2. Descriptive statistics and tail behaviour

Means alone are insufficient for skewed trip data, so we report medians, percentiles, maxima, skewness, and excess kurtosis. Plots stop at the 99.5th percentile only for readability; maxima are printed and every observation remains in every calculation.

In [3]:
descriptive = descriptive_statistics(training)
display(descriptive[[
    "label", "count", "missing", "mean", "median", "p95", "p99",
    "maximum", "skewness", "excess_kurtosis",
]].style.format(precision=3))

,label,count,missing,mean,median,p95,p99,maximum,skewness,excess_kurtosis
0,Fare amount USD,40000,0,16.031,12.100,40.800,65.300,204.600,2.753,11.622
1,Trip distance miles,40000,0,2.550,1.530,9.300,15.500,54.830,3.402,16.801
2,Duration minutes,40000,0,14.217,11.583,32.768,50.384,1437.050,51.752,3948.135
3,Passenger count,40000,0,1.262,1.000,3.000,4.000,6.000,3.318,12.992
4,Pickup hour,40000,0,14.450,15.000,22.000,23.000,23.000,-0.662,0.021
5,Pickup hour sine,40000,0,-0.328,-0.500,0.866,1.000,1.000,0.702,-0.840
6,Pickup hour cosine,40000,0,-0.197,-0.259,0.966,1.000,1.000,0.435,-1.247
7,Weekend indicator,40000,0,0.262,0.000,1.000,1.000,1.000,1.080,-0.834


In [4]:
variables = ["fare_amount", "trip_distance", "duration_min", "passenger_count"]
labels = ["Fare amount USD", "Trip distance miles", "Duration minutes", "Passenger count"]
fig, axes = plt.subplots(2, 2, figsize=(11, 7.5))
for axis, variable, label in zip(axes.flat, variables, labels, strict=True):
    upper = training[variable].quantile(0.995)
    sns.histplot(training.loc[training[variable].le(upper), variable], bins=45, ax=axis, color="#315C7C")
    axis.set_title(label)
    axis.set_xlabel(f"Shown to p99.5={upper:.2f}; full maximum={training[variable].max():.2f}")
    axis.set_ylabel("Training trips")
plt.tight_layout()
plt.show()

/var/folders/69/dxynlnqj6413x3bgy4c63zhc0000gn/T/ipykernel_88097/1717112737.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Interpretation.** Fare, distance, and duration are right-skewed. Duration is exceptionally heavy-tailed: its median is about 11.58 minutes but its maximum is 1,437.05 minutes. Such observations may be valid, interrupted, or erroneous; they are retained so robust regression can assess compatibility with the dominant pattern.

## 3. Fare relationships

Hexagonal density plots are preferable to ordinary scatterplots because 40,000 points overlap heavily. Darker cells contain more trips.

In [5]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
fare_limit = training[TARGET].quantile(0.995)
for axis, variable, label in zip(
    axes, ["trip_distance", "duration_min"], ["Trip distance miles", "Duration minutes"], strict=True
):
    x_limit = training[variable].quantile(0.995)
    visible = training.loc[training[variable].le(x_limit) & training[TARGET].le(fare_limit)]
    density = axis.hexbin(visible[variable], visible[TARGET], gridsize=48, mincnt=1, bins="log", cmap="Blues")
    axis.set(xlabel=label, ylabel="Fare amount USD", title=f"Fare versus {label.lower()}")
    fig.colorbar(density, ax=axis, label="Log trip density")
plt.tight_layout()
plt.show()

/var/folders/69/dxynlnqj6413x3bgy4c63zhc0000gn/T/ipykernel_88097/2797771601.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Distance has a narrow positive relationship with fare. The fare-duration cloud spreads outward for longer trips, suggesting nonconstant error variance. Parallel fare bands probably reflect tariff and operational details not represented by the current predictors.

## 4. Pearson versus Spearman association

- **Pearson** measures linear association and is sensitive to extreme magnitudes.
- **Spearman** computes Pearson correlation on ranks, measuring monotonic association with less sensitivity to magnitude.

A large difference between them is evidence that the tail geometry matters.

In [6]:
pearson, spearman = correlation_tables(training)
print("Pearson correlations")
display(pearson.round(3))
print("Spearman rank correlations")
display(spearman.round(3))

fig, axis = plt.subplots(figsize=(8, 6))
sns.heatmap(pearson, annot=True, fmt=".2f", cmap="vlag", center=0, vmin=-1, vmax=1, ax=axis)
plt.title("Pearson correlations in the training data")
plt.tight_layout()
plt.show()

Pearson correlations


,fare_amount,trip_distance,duration_min,passenger_count,pickup_hour_sin,pickup_hour_cos,is_weekend
fare_amount,1.000,0.953,0.494,0.023,-0.009,0.018,-0.032
trip_distance,0.953,1.000,0.405,0.019,0.011,0.085,0.004
duration_min,0.494,0.405,1.000,0.022,-0.027,-0.039,-0.039
passenger_count,0.023,0.019,0.022,1.000,-0.047,0.053,0.085
pickup_hour_sin,-0.009,0.011,-0.027,-0.047,1.000,-0.239,0.016
pickup_hour_cos,0.018,0.085,-0.039,0.053,-0.239,1.000,0.072
is_weekend,-0.032,0.004,-0.039,0.085,0.016,0.072,1.000


Spearman rank correlations


,fare_amount,trip_distance,duration_min,passenger_count,pickup_hour_sin,pickup_hour_cos,is_weekend
fare_amount,1.000,0.909,0.967,0.032,-0.012,0.015,-0.040
trip_distance,0.909,1.000,0.816,0.026,0.020,0.113,0.015
duration_min,0.967,0.816,1.000,0.030,-0.048,-0.038,-0.066
passenger_count,0.032,0.026,0.030,1.000,-0.046,0.059,0.094
pickup_hour_sin,-0.012,0.020,-0.048,-0.046,1.000,-0.304,0.032
pickup_hour_cos,0.015,0.113,-0.038,0.059,-0.304,1.000,0.057
is_weekend,-0.040,0.015,-0.066,0.094,0.032,0.057,1.000


/var/folders/69/dxynlnqj6413x3bgy4c63zhc0000gn/T/ipykernel_88097/355279548.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Fare-distance correlation is high under both measures (Pearson 0.953, Spearman 0.909). Fare-duration correlation changes from Pearson 0.494 to Spearman 0.967. Ordinary trips therefore follow a very strong monotonic duration-fare pattern, while a few extreme durations do not carry proportionally extreme fares.

## 5. Variance inflation factors

For predictor \(j\),

\[
VIF_j=\frac{1}{1-R_j^2},
\]

where \(R_j^2\) comes from regressing predictor \(j\) on all other predictors. Large VIF means the slope is difficult to distinguish from linear combinations of the remaining predictors.

In [7]:
vif = vif_table(training)
display(vif.style.format({"vif": "{:.3f}"}))

,term,vif
0,const,6.141
1,trip_distance,1.215
2,duration_min,1.210
3,passenger_count,1.012
4,pickup_hour_sin,1.067
5,pickup_hour_cos,1.086
6,is_weekend,1.015


All predictor VIFs are between about 1.01 and 1.21. Conventional linear multicollinearity is therefore low. This does not contradict the large Spearman distance-duration correlation because VIF concerns linear predictability and is affected by the extreme durations.

## 6. Diagnostic OLS fit and HC3 uncertainty

The provisional additive conditional-mean model is

\[
fare_i=\beta_0+\beta_1 distance_i+\beta_2 duration_i+\beta_3 passengers_i
+\beta_4\sin(hour_i)+\beta_5\cos(hour_i)+\beta_6 weekend_i+\varepsilon_i.
\]

OLS minimizes all squared residuals, so large residuals and high-leverage predictor values can dominate it. HC3 standard errors inflate each squared residual according to leverage and are more credible under heteroskedasticity than conventional OLS standard errors.

In [8]:
ols_model, ols_hc3 = fit_diagnostic_ols(training)
coefficients = coefficient_table(ols_model, ols_hc3)
display(coefficients.style.format({
    "coefficient": "{:.4f}",
    "standard_error_conventional": "{:.4f}",
    "standard_error_hc3": "{:.4f}",
    "hc3_ci_lower_95": "{:.4f}",
    "hc3_ci_upper_95": "{:.4f}",
    "hc3_p_value": "{:.3g}",
}))

,term,coefficient,standard_error_conventional,standard_error_hc3,hc3_ci_lower_95,hc3_ci_upper_95,hc3_p_value
0,const,5.2406,0.0407,0.2853,4.6815,5.7998,4.64e-75
1,trip_distance,3.6218,0.0059,0.0958,3.4339,3.8096,4.83e-307
2,duration_min,0.0824,0.0010,0.0392,0.0055,0.1593,0.0357
3,passenger_count,0.1380,0.0246,0.0360,0.0674,0.2086,0.000127
4,pickup_hour_sin,-0.5781,0.0270,0.0611,-0.6979,-0.4583,3.23e-21
5,pickup_hour_cos,-1.0891,0.0253,0.0876,-1.2609,-0.9173,2.18e-35
6,is_weekend,-0.7448,0.0376,0.0653,-0.8728,-0.6169,4.12e-30


Holding other included predictors fixed, the provisional slopes are about $3.62 per mile and $0.08 per minute. They are conditional associations, not causal effects. HC3 errors are roughly 16 times the conventional distance error and 39 times the conventional duration error, demonstrating how misleading homoskedastic uncertainty would be.

## 7. Fit and assumption diagnostics

In [9]:
diagnostics, influence_summary, binned, residual_scale, influential = diagnostic_tables(training, ols_model)
diagnostic_values = diagnostics.set_index("diagnostic")["value"]

key_diagnostics = [
    "training_r_squared", "training_adjusted_r_squared", "training_mae", "training_rmse",
    "training_median_absolute_error", "residual_skewness", "residual_kurtosis",
    "breusch_pagan_lm", "breusch_pagan_lm_p_value", "ramsey_reset_f",
    "ramsey_reset_p_value", "jarque_bera", "jarque_bera_p_value",
    "shapiro_wilk_sample_5000", "shapiro_wilk_p_value", "durbin_watson",
]
display(diagnostics.loc[diagnostics["diagnostic"].isin(key_diagnostics)].style.format({"value": "{:.6g}"}))

,diagnostic,value
2,training_r_squared,0.92693
3,training_adjusted_r_squared,0.926919
4,training_mae,2.07243
5,training_rmse,3.28215
6,training_median_absolute_error,1.57642
9,residual_skewness,-3.21356
10,residual_kurtosis,187.397
11,breusch_pagan_lm,33831.5
12,breusch_pagan_lm_p_value,0
15,ramsey_reset_f,3692.56


### How to read the tests

- **Breusch–Pagan:** null hypothesis is constant residual variance. The enormous statistic rejects it.
- **Ramsey RESET:** tests whether powers of fitted values add explanatory information. Rejection indicates omitted nonlinearity or interactions.
- **Jarque–Bera and Shapiro–Wilk:** test residual normality. Both reject strongly; graphical magnitude matters because \(n=40{,}000\) makes tiny deviations significant.
- **Durbin–Watson:** approximately 2 in the stored random order. This does not prove temporal independence because rows are not one continuous time series.

The in-sample \(R^2=0.9269\), MAE=$2.07, and RMSE=$3.28 describe training fit only—not test performance.

In [10]:
fitted = np.asarray(ols_model.fittedvalues)
residuals = np.asarray(ols_model.resid)
rng = np.random.default_rng(42)
qq_residuals = rng.choice(residuals, size=5_000, replace=False)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
density = axes[0].hexbin(fitted, residuals, gridsize=55, mincnt=1, bins="log", cmap="Blues")
axes[0].axhline(0, color="#8B1E1E")
axes[0].set(xlabel="OLS fitted fare USD", ylabel="Residual USD", title="Residuals versus fitted values")
fig.colorbar(density, ax=axes[0], label="Log trip density")
stats.probplot(qq_residuals, dist="norm", plot=axes[1])
axes[1].set_title("Normal Q-Q plot from 5,000 residuals")
plt.tight_layout()
plt.show()

/var/folders/69/dxynlnqj6413x3bgy4c63zhc0000gn/T/ipykernel_88097/3990214784.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


The funnel-shaped residual plot confirms increasing variance. The Q-Q plot has heavy asymmetric tails, consistent with residual skewness −3.21 and kurtosis 187.4. Normal-theory OLS inference is therefore unsuitable without qualification.

## 8. Functional form using binned residuals

If the conditional mean is well specified, mean residuals should remain near zero across predictor values. Quantile bins provide a readable nonparametric check without fitting a new model.

In [11]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for axis, variable, label in zip(
    axes, ["trip_distance", "duration_min"], ["Trip distance miles", "Duration minutes"], strict=True
):
    subset = binned.loc[binned["variable"].eq(variable)]
    axis.plot(subset["predictor_mean"], subset["residual_mean"], marker="o", color="#315C7C")
    axis.fill_between(
        subset["predictor_mean"], subset["residual_mean_ci_lower_95"],
        subset["residual_mean_ci_upper_95"], color="#315C7C", alpha=0.18,
    )
    axis.axhline(0, color="#8B1E1E")
    axis.set(xlabel="Mean within quantile bin", ylabel="Mean OLS residual USD", title=label)
plt.tight_layout()
plt.show()

display(residual_scale.style.format(precision=3))

/var/folders/69/dxynlnqj6413x3bgy4c63zhc0000gn/T/ipykernel_88097/3036842458.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,fitted_decile,count,fitted_mean,mean_absolute_residual,median_absolute_residual,residual_rmse
0,1,4000,7.466,1.638,1.611,1.913
1,2,4000,8.863,1.752,1.704,2.067
2,3,4000,9.776,1.654,1.544,2.009
3,4,4000,10.683,1.680,1.411,2.246
4,5,4000,11.686,1.633,1.327,2.275
5,6,4000,12.920,1.743,1.278,2.573
6,7,4000,14.495,1.947,1.311,2.944
7,8,4000,16.931,2.318,1.496,3.582
8,9,4000,22.357,2.872,1.922,4.324
9,10,4000,45.134,3.487,2.665,6.223


Distance residual means curve from negative to positive and back to negative. Duration residual means increase from approximately −$2.48 to +$4.64. Residual RMSE rises from $1.91 in the lowest fitted-fare decile to $6.22 in the highest. The additive linear mean is useful but incomplete, and the variance is clearly nonconstant.

## 9. Leverage, studentized residuals, and Cook's distance

- **Leverage** measures how unusual a predictor vector is. Screening threshold: \(2p/n\).
- **Studentized residual** measures response disagreement relative to an estimated residual scale. Screening rule: absolute value above 3.
- **Cook's distance** estimates how much the fitted coefficient vector changes when one row is removed. Screening threshold: \(4/n\).

These are diagnostic screens, never automatic deletion rules.

In [12]:
display(influence_summary.T.rename(columns={0: "Value"}).style.format(precision=6))
display(influential.head(10).style.format(precision=3))

,Value
training_observations,40000.000000
parameter_count,7.000000
leverage_threshold_2p_over_n,0.000350
high_leverage_count,2038.000000
high_leverage_fraction,0.050950
cook_threshold_4_over_n,0.000100
cook_above_threshold_count,1663.000000
cook_above_threshold_fraction,0.041575
absolute_studentized_residual_above_3_count,575.000000
absolute_studentized_residual_above_3_fraction,0.014375


,modeling_row_id,source_row_id,fare_amount,trip_distance,duration_min,passenger_count,pickup_hour,is_weekend,fitted_fare,residual,studentized_residual_internal,leverage,cooks_distance
39938,39938,2429744,7.900,1.230,1437.050,1.000,20,0,128.226,-120.326,-40.791,0.192,56.622
21045,21045,660951,7.200,0.520,1436.367,1.000,11,0,126.544,-119.344,-40.454,0.192,55.633
41197,41197,251108,8.600,1.410,1412.117,5.000,12,1,127.763,-119.163,-40.230,0.186,52.728
48214,48214,433589,24.000,4.400,1413.850,2.000,19,0,138.253,-114.253,-38.529,0.184,47.772
41908,41908,1419099,42.200,10.700,737.383,1.000,11,1,105.062,-62.862,-19.619,0.047,2.720
27108,27108,1349531,4.400,0.190,511.667,1.000,15,0,49.415,-45.015,-13.880,0.024,0.672
24930,24930,347010,82.100,19.550,367.583,1.000,18,0,107.057,-24.957,-7.642,0.010,0.084
8307,8307,2268584,136.700,17.470,148.950,3.000,7,0,80.926,55.774,17.007,0.002,0.076
40734,40734,250728,117.100,16.230,138.967,1.000,13,1,76.070,41.030,12.509,0.001,0.033
44488,44488,2456086,40.100,20.900,35.050,1.000,5,0,83.122,-43.022,-13.114,0.001,0.027


There are 2,038 high-leverage rows, 1,663 rows above the Cook's-distance screen, and 575 with absolute internally studentized residual above 3. The four largest Cook's distances belong to reported durations around 1,412–1,437 minutes with relatively small fares. They remain in the data for robust modelling and possible audit review.

## 10. Frozen model-specification decision

The primary comparison will keep the same predictors for OLS and LTS so that the estimator—not a different feature set—is being compared:

1. Distance, duration, passenger count, pickup-hour sine/cosine, and weekend status.
2. Fit all scaling parameters on the training records only and apply them unchanged to the test set.
3. Report OLS with HC3 uncertainty.
4. Fit raw and explicitly reweighted FAST-LTS at \(\alpha=0.75\); check \(0.50\) and \(0.90\).
5. Keep extreme observations. Call robustly flagged rows **audit candidates**, not fraud.
6. After robust fitting, repeat binned-residual diagnostics. If curvature remains, add a training-prespecified nonlinear sensitivity model without consulting test outcomes.

The next phase is clean OLS and FAST-LTS fitting, followed by one evaluation on the untouched test set. The controlled +$100 contamination experiment comes only after clean results are frozen.

## Short oral explanation

> The training data have a strong fare-distance relationship, but ordinary least squares assumptions are not credible. Residual variance grows with predicted fare, residual means show curvature, residual tails are highly non-normal, and a few extreme durations have enormous influence. Low VIF values show this is not mainly a multicollinearity problem. We therefore retain an interpretable common linear specification, use HC3 uncertainty for OLS, and compare it with resistant FAST-LTS while preserving all extreme records for audit assessment.